# Current legal dataset report

This report lists the law and legal-document titles currently represented in the processed article dataset. It also counts articles, identifies document types, and writes a CSV and JSON report for review.

In [1]:
import csv
import json
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
REPORT_DIR = DATA_DIR / 'processed' / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

articles = [
    json.loads(line)
    for line in INPUT_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'Article records: {len(articles):,}')
print(f'Input: {INPUT_PATH}')

Article records: 2,744
Input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/articles_normalized.jsonl


In [2]:
documents = {}
for article in articles:
    document_id = str(article.get('document_id', '')).strip()
    title = str(article.get('document_title') or '').strip()
    item = documents.setdefault(document_id, {
        'document_id': document_id,
        'title': title,
        'document_type': article.get('document_type'),
        'source_url': article.get('source_url'),
        'article_count': 0,
        'article_ids': [],
    })
    item['article_count'] += 1
    item['article_ids'].append(article.get('id'))
    if not item['title'] and title:
        item['title'] = title

report = sorted(documents.values(), key=lambda item: (item['title'], item['document_id']))
for item in report:
    item['title_status'] = 'missing' if not item['title'] else 'present'
    item['duplicate_article_ids'] = len(item['article_ids']) != len(set(item['article_ids']))

print(f'Documents: {len(report):,}')
print(f'Documents with missing titles: {sum(item["title_status"] == "missing" for item in report):,}')
print(f'Total articles: {sum(item["article_count"] for item in report):,}')

Documents: 90
Documents with missing titles: 0
Total articles: 2,744


In [3]:
csv_path = REPORT_DIR / 'law_dataset_report.csv'
json_path = REPORT_DIR / 'law_dataset_report.json'
fieldnames = ['document_id', 'title', 'document_type', 'article_count', 'title_status', 'source_url']
with csv_path.open('w', encoding='utf-8-sig', newline='') as output:
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({key: item.get(key) for key in fieldnames} for item in report)
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'CSV report: {csv_path}')
print(f'JSON report: {json_path}')

CSV report: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/reports/law_dataset_report.csv
JSON report: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/reports/law_dataset_report.json


In [4]:
print('Law and legal-document titles')
print('=' * 100)
for number, item in enumerate(report, start=1):
    print(f"{number:>4}. [{item['document_id']}] {item['title'] or '[MISSING TITLE]'} — {item['article_count']} articles")

Law and legal-document titles
   1. [11390] Convention against Discrimination in Education - អនុសញ្ញាប្រឆាំងនឹងការរើសអើងក្នុងការអប់រំ — 19 articles
   2. [11167] កិច្ចព្រមព្រៀង ទីក្រុងឡូការណូ ស្តីពីការបង្កើតចំណាត់ថ្នាក់អន្តរជាតិចំពោះគំនូរឧស្សាហកម្ម — 15 articles
   3. [11168] កិច្ចព្រមព្រៀង ស្តីពីទិដ្ឋភាពពាណិជ្ជកម្មទាក់ទងនឹងកម្មសិទ្ធិបញ្ញា (កិច្ចព្រមព្រៀង TRIPS 1994) — 73 articles
   4. [11165] កិច្ចព្រមព្រៀងទីក្រុងស្រ្តាស្បួរ ស្តីពីចំណាត់ថ្នាក់ប្រកាសនីយបត្រតក្កកម្មអន្តរជាតិ — 17 articles
   5. [12360] ក្រមសីលធម៌សម្រាប់មន្ត្រីប្រតិបត្តិច្បាប់ — 8 articles
   6. [1202] ច្បាប់ បន្ថែមលើកទី២ទៅលើច្បាប់ស្តីពីការបោះឆ្នោតជ្រើសតាំងតំណាងរាស្ត្រ — 8 articles
   7. [1204] ច្បាប់ បន្ថែមលើកទី៣ទៅលើច្បាប់ស្តីពីការបោះឆ្នោតជ្រើសតាំងតំណាងរាស្ត្រ — 2 articles
   8. [12315] ច្បាប់ ស្ដីពីប្រាក់ឈ្នួលអប្បបរមា — 30 articles
   9. [10813] ច្បាប់ ស្តីពីការបង្កើតក្រសួងសេដ្ឋកិច្ច និងហិរញ្ញវត្ថុ — 6 articles
  10. [11943] ច្បាប់ ស្តីពីការរៀបចំ និងការប្រព្រឹត្តទៅ នៃគណៈរដ្ឋមន្ត្រី (២០១៨) — 59 articles
  11. [11919] ច